# 島ごとの在庫列挙 (GUID 特定用スパイク)

**目的**: Anno 117 セーブの `AreaInfo > <1> > AreaEconomy > StorageTrends` を全件 walk し，
島 × 物資 GUID × 最新在庫量 を pandas DataFrame に落とす．
書記長がゲーム内で見えてる物資量と突き合わせて GUID → 品名を同定する作業用．

**データソース選定の経緯**:
- `LogisticNode > Storage` は本セーブで 1 建物分しか存在しない (3 行) — 採用不可
- `AreaEconomy > StrgLrg` は 496B の packed binary で format 未解読 — 採用不可
- `AreaEconomy > StorageTrends` は `<ProductGUID> > <1> > Points > [size=120 の時系列]` の
  きれいな構造で 880 行取れる → **採用**．`latest` = 最新 sample = 現在在庫量．

## 事前

```bash
pip install jupyterlab
jupyter lab notebooks/island_inventory.ipynb
```


In [ ]:
from __future__ import annotations

import struct
from pathlib import Path

import pandas as pd

from anno_save_analyzer.parser.pipeline import extract_inner_filedb
from anno_save_analyzer.parser.filedb import (
    EventKind,
    detect_version,
    extract_sessions,
    iter_dom,
    parse_tag_section,
)

SAVE = Path('..') / 'sample_anno117.a8s'
assert SAVE.exists(), f'save not found: {SAVE.resolve()}'
outer = extract_inner_filedb(str(SAVE))
outer_version = detect_version(outer)
outer_section = parse_tag_section(outer, outer_version)
sessions = extract_sessions(outer, version=outer_version, tag_section=outer_section)
print(f'sessions: {len(sessions)}  sizes: {[len(s) for s in sessions]}')


## 1. 島メタ一覧

`AreaInfo > <1>` 直下の attrib から (CityName, OwnerProfile, WasEverOwned 等) を抽出．
書記長の OwnerProfile は `41` (Anno 117 でのプレイヤー ID)．


In [ ]:
def _read_int(buf: bytes) -> int | None:
    if len(buf) == 4:
        return struct.unpack('<i', buf)[0]
    if len(buf) == 8:
        return struct.unpack('<q', buf)[0]
    return None


def collect_islands(inner: bytes) -> list[dict]:
    iv = detect_version(inner)
    isec = parse_tag_section(inner, iv)
    area_info_id = next((tid for tid, n in isec.tags.entries.items() if n == 'AreaInfo'), None)
    if area_info_id is None:
        return []
    rows: list[dict] = []
    stack: list[int] = []
    in_area_depth: int | None = None
    entry_depth: int | None = None
    cur: dict | None = None
    entry_idx = 0
    for ev in iter_dom(inner, iv, tag_section=isec):
        if ev.kind is EventKind.TAG:
            stack.append(ev.id_)
            if ev.id_ == area_info_id and in_area_depth is None:
                in_area_depth = len(stack)
            elif in_area_depth is not None and len(stack) == in_area_depth + 1:
                entry_idx += 1
                entry_depth = len(stack)
                cur = {'entry_idx': entry_idx}
            continue
        if ev.kind is EventKind.ATTRIB:
            if cur is not None and entry_depth is not None and len(stack) == entry_depth:
                name = ev.name or f'<{ev.id_}>'
                if name == 'CityName':
                    cur['city_name'] = ev.content.decode('utf-16-le', errors='replace').rstrip('\x00')
                elif name == 'CityNameGuid':
                    cur['city_name_guid'] = _read_int(ev.content)
                elif name == 'OwnerProfile':
                    cur['owner_profile'] = _read_int(ev.content)
                elif name == 'WasEverOwned':
                    cur['was_ever_owned'] = bool(ev.content[0]) if ev.content else False
                elif name == 'VassalOwner':
                    cur['vassal_owner'] = _read_int(ev.content)
                elif name == 'OriginalOwner':
                    cur['original_owner'] = _read_int(ev.content)
            continue
        if not stack:
            continue
        if entry_depth is not None and len(stack) == entry_depth:
            if cur is not None:
                rows.append(cur)
            cur = None
            entry_depth = None
        if in_area_depth is not None and len(stack) == in_area_depth:
            in_area_depth = None
        stack.pop()
    return rows


all_islands = []
for sidx, inner in enumerate(sessions):
    if not inner:
        continue
    for row in collect_islands(inner):
        row['session'] = sidx
        all_islands.append(row)
islands_df = pd.DataFrame(all_islands)
cols = ['session', 'entry_idx', 'city_name', 'city_name_guid',
        'owner_profile', 'vassal_owner', 'original_owner', 'was_ever_owned']
for c in cols:
    if c not in islands_df.columns:
        islands_df[c] = None
islands_df = islands_df[cols]
print(f'total island entries (player + NPC): {len(islands_df)}')
player_df = islands_df[islands_df.city_name.notna()].reset_index(drop=True)
print(f'player-owned islands (CityName 持ち): {len(player_df)}')
player_df


## 2. StorageTrends → 島×物資 在庫スナップショット

各 `AreaInfo > <1>` (player island のみ) 配下の
`AreaEconomy > StorageTrends > <ProductGUID=anon attrib> > <1> > Points` を展開する．

`Points` は `size=120` の固定長リングバッファ (サンプル履歴)．
`latest` = 最新サンプル = 現在在庫に相当，`peak` = 120 サンプル中の最大値．


In [ ]:
def collect_storage_trends(inner: bytes) -> list[dict]:
    iv = detect_version(inner)
    isec = parse_tag_section(inner, iv)
    area_info_id = next((tid for tid, n in isec.tags.entries.items() if n == 'AreaInfo'), None)
    storage_trends_id = next(
        (tid for tid, n in isec.tags.entries.items() if n == 'StorageTrends'), None)
    points_id = next((tid for tid, n in isec.tags.entries.items() if n == 'Points'), None)
    if area_info_id is None or storage_trends_id is None or points_id is None:
        return []

    rows: list[dict] = []
    stack: list[int] = []
    in_area_depth: int | None = None
    entry_depth: int | None = None
    city: str | None = None
    in_trends_depth: int | None = None
    in_points_depth: int | None = None
    current_guid: int | None = None
    points: list[int] = []

    for ev in iter_dom(inner, iv, tag_section=isec):
        if ev.kind is EventKind.TAG:
            stack.append(ev.id_)
            if ev.id_ == area_info_id and in_area_depth is None:
                in_area_depth = len(stack)
            elif in_area_depth is not None and len(stack) == in_area_depth + 1:
                entry_depth = len(stack)
                city = None
            elif ev.id_ == storage_trends_id and entry_depth is not None:
                in_trends_depth = len(stack)
            elif ev.id_ == points_id and in_trends_depth is not None:
                in_points_depth = len(stack)
                points = []
            continue
        if ev.kind is EventKind.ATTRIB:
            if entry_depth is not None and len(stack) == entry_depth and ev.name == 'CityName':
                city = ev.content.decode('utf-16-le', errors='replace').rstrip('\x00')
            if (in_trends_depth is not None
                    and len(stack) == in_trends_depth
                    and ev.name is None
                    and len(ev.content) == 4):
                current_guid = struct.unpack('<i', ev.content)[0]
            if (in_points_depth is not None
                    and len(stack) == in_points_depth
                    and ev.name is None
                    and len(ev.content) == 4):
                points.append(struct.unpack('<i', ev.content)[0])
            continue
        if not stack:
            continue
        depth = len(stack)
        if in_points_depth is not None and depth == in_points_depth:
            if city and current_guid is not None and points:
                rows.append({
                    'city_name': city,
                    'product_guid': current_guid,
                    'latest': points[-1],
                    'peak': max(points),
                    'mean': sum(points) / len(points),
                    'samples': len(points),
                })
            in_points_depth = None
        if in_trends_depth is not None and depth == in_trends_depth:
            in_trends_depth = None
        if entry_depth is not None and depth == entry_depth:
            entry_depth = None
        if in_area_depth is not None and depth == in_area_depth:
            in_area_depth = None
        stack.pop()
    return rows


trend_rows = []
for sidx, inner in enumerate(sessions):
    if not inner:
        continue
    for row in collect_storage_trends(inner):
        row['session'] = sidx
        trend_rows.append(row)
trends_df = pd.DataFrame(trend_rows)
trends_df = trends_df[['session', 'city_name', 'product_guid',
                        'latest', 'peak', 'mean', 'samples']]
print(f'total trend rows: {len(trends_df)}')
print(f'unique cities: {trends_df.city_name.nunique()}')
print(f'unique product GUIDs: {trends_df.product_guid.nunique()}')
trends_df.head(15)


## 3. 島 × 物資 の pivot (latest)

書記長がゲーム内で見えてる各島の在庫量と照合する用．
行=CityName, 列=ProductGUID, 値=latest．0 が多い列は削る．


In [ ]:
pivot_latest = trends_df.pivot_table(
    index='city_name',
    columns='product_guid',
    values='latest',
    aggfunc='first',
    fill_value=0,
)
# どの島でも 0 の列を削る
nonzero_cols = pivot_latest.columns[pivot_latest.sum(axis=0) > 0]
pivot_latest = pivot_latest[nonzero_cols]
pivot_latest


## 4. 1 島ごとに wide → long で top N

書記長がゲーム画面と照合しやすい形．`latest > 0` に絞って大きい順．


In [ ]:
def show_city(city: str, top_n: int = 30) -> pd.DataFrame:
    sub = trends_df[(trends_df.city_name == city) & (trends_df.latest > 0)]
    return sub.sort_values('latest', ascending=False).head(top_n).reset_index(drop=True)


# 大阪民国 (Latium 最大都市)
show_city('大阪民国', top_n=30)


In [ ]:
show_city('ジョウト地方', top_n=20)


In [ ]:
show_city('シベリア', top_n=20)


## 5. CSV にエクスポート

`notebooks/out/` に書き出し．DuckDB / Excel 等で GUID 特定作業に使う．


In [ ]:
out = Path('out')
out.mkdir(exist_ok=True)
islands_df.to_csv(out / 'islands.csv', index=False)
trends_df.to_csv(out / 'storage_trends.csv', index=False)
pivot_latest.to_csv(out / 'pivot_latest.csv')
print('wrote:')
for p in sorted(out.iterdir()):
    print(f'  {p} ({p.stat().st_size:,} B)')
